# Week 4 — Date Functions: Reading Time Out of Text
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Extract the year and the month from a SQLite timestamp with `strftime()`, and group by
  the part you extracted to turn a table of 99,441 individual orders into a trend
- Compute the number of days between two dates with `julianday()`, and use it to answer
  operational questions like *how long does delivery actually take?*
- Compare two date columns against each other to measure a promise against reality —
  counting the orders that arrived later than the date the customer was told

---
*Session timing: 2 hours | AI assistance: DeepSeek (introduced this week)*

### Run this first

The cell below loads all eight Olist tables into a SQLite database and connects the
`%%sql` magic to it. It is the same setup cell you ran on Wednesday — run it once,
top to bottom, before any query cell.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71

## Why this matters

Every one of the 99,441 rows in the Olist `orders` table carries five timestamps: when the
order was placed, when payment was approved, when the seller handed it to the carrier, when
it reached the customer, and the delivery date the customer was *promised* at checkout.
That is a complete operational history of the business, sitting there in five columns.

And so far you have not been able to touch any of it. Every query you have written since
Week 1 has treated `order_purchase_timestamp` as an opaque blob of characters — you could
sort by it and you could compare it, but you could not ask *"how many orders in November?"*,
because "November" is not a value in that column. The column holds
`2017-11-24 09:22:16`, and the month is buried three characters deep inside a string.

That is the gap today closes. Olist's real questions are time questions: is the business
growing, when is the seasonal peak that we must staff for, how long does a customer wait,
and how often do we break the delivery promise we made. Two SQLite functions — `strftime()`
to pull a piece out of a timestamp, and `julianday()` to subtract one timestamp from
another — turn those five text columns into answers.

## 1. Dates in SQLite are text — and `strftime()` is the way in

SQLite has no dedicated date type. None. A timestamp is stored as an ordinary `TEXT` value
in the ISO 8601 format `YYYY-MM-DD HH:MM:SS`, and SQLite treats it exactly like any other
string — which is why `ORDER BY order_purchase_timestamp` has worked correctly for you all
along. That is not luck. ISO 8601 is deliberately designed so that alphabetical order and
chronological order are the same order: `2017-03-01` sorts before `2017-11-24` as text for
precisely the same reason it comes before it in time.

`strftime()` ("string format time") is the function that reaches into that string and pulls
out one piece of it. You call it with two arguments — a **format code** saying which piece
you want, and the column you want it from — and it hands you back that piece, also as text.
`strftime('%Y', order_purchase_timestamp)` returns `'2017'`. The codes you will use most are
`%Y` (4-digit year), `%m` (2-digit month), `%d` (day of month), `%H` (hour), and `%w`
(day of the week, `0` = Sunday through `6` = Saturday).

The query below proves both halves of that claim at once. `typeof()` reports what SQLite
thinks a value actually is, and beside it you can watch `strftime()` carve the same
timestamp into four different pieces. These are the oldest orders in the dataset — note
where the data begins.

In [ ]:
%%sql
-- typeof() tells you how SQLite really stores the value; strftime() carves
-- pieces out of it. The oldest orders in the dataset are from September 2016.
SELECT order_id,
       order_purchase_timestamp,
       typeof(order_purchase_timestamp)          AS stored_as,
       strftime('%Y', order_purchase_timestamp)  AS year,
       strftime('%m', order_purchase_timestamp)  AS month,
       strftime('%d', order_purchase_timestamp)  AS day,
       strftime('%H', order_purchase_timestamp)  AS hour
FROM orders
WHERE order_purchase_timestamp IS NOT NULL
ORDER BY order_purchase_timestamp
LIMIT 5

## 2. Grouping by an extracted part — orders per year

Here is the idea that makes `strftime()` powerful, and it is one you already know in another
costume. On Wednesday, `CASE WHEN` invented a column that did not exist in the table
(`status_group`) and you grouped by it. `strftime('%Y', ...)` does the same thing: it
invents a `year` column out of a timestamp, and `GROUP BY year` then collapses tens of
thousands of individual orders into one row per year.

Give the extracted piece a name with `AS`, and every clause you know can use that name —
`GROUP BY`, `ORDER BY`, even `WHERE` (with one caveat you will meet in section 3). Without
the alias you would have to repeat the whole `strftime(...)` expression in the `GROUP BY`,
which works but reads badly.

Read the result carefully, because it contains a trap that catches almost every analyst
once. The three numbers are 329, 45,101 and 54,011 — and the tempting story is *"explosive
growth, then a slowdown."* That story is wrong. The dataset simply **begins in September
2016**, so 2016 is not a year at all; it is about four months, and thin ones. A partial
period sitting in a table next to complete periods looks exactly like a real number, and
comparing it to them produces confident nonsense. Always ask what window your data actually
covers before you read a trend out of it.

In [ ]:
%%sql
-- Orders per year. strftime('%Y', ...) invents a `year` column out of the
-- timestamp; GROUP BY then collapses 99,441 orders into one row per year.
-- Expected: 2016 -> 329 | 2017 -> 45,101 | 2018 -> 54,011
-- NOTE: 2016 is an INCOMPLETE year (data starts September 2016) — do not
-- compare it directly against 2017 or 2018.
SELECT strftime('%Y', order_purchase_timestamp) AS year,
       COUNT(*) AS order_count
FROM orders
GROUP BY year
ORDER BY year

## 3. Zooming in — monthly seasonality across 2017

A year is too coarse to run a business on. Warehouse staffing, seller onboarding and
customer-support rotas are all planned month by month, so the useful question is not
*"how big was 2017?"* but *"which months of 2017 were big?"*

`strftime('%Y-%m', ...)` returns `'2017-11'` — year and month glued together in one value.
That combined form matters: grouping on `%m` alone would merge every November in the
dataset into a single bucket, so 2017's November and 2018's November would be added
together and the trend would disappear. Keeping the year in the label keeps each month
distinct, and it sorts chronologically for free, again thanks to ISO ordering.

Two details in the query below are worth pausing on. First, the `WHERE` clause filters with
`strftime('%Y', ...) = '2017'` — note the **quotes**. `strftime()` returns TEXT, so the
value it produces must be compared against the *string* `'2017'`, never the bare number
`2017`. Second, `WHERE` is written in terms of the full `strftime(...)` expression rather
than the `month` alias, because SQL evaluates `WHERE` before the `SELECT` list exists.

The shape of the result is the point. Orders climb steadily through the year, and then
**November 2017 spikes to 7,544** — roughly double an ordinary month, and a third higher
than any month before it. That is Black Friday, which in Brazil is the single largest
retail event of the year. December falls back to 5,673: the peak is a genuine event, not
a new baseline.

In [ ]:
%%sql
-- Monthly orders for 2017 only. '%Y-%m' keeps the year attached to the month,
-- so 2017-11 and 2018-11 never merge into one bucket.
-- Expected (12 rows): 2017-01 800 | 2017-02 1,780 | 2017-03 2,682
--   | 2017-04 2,404 | 2017-05 3,700 | 2017-06 3,245 | 2017-07 4,026
--   | 2017-08 4,331 | 2017-09 4,285 | 2017-10 4,631 | 2017-11 7,544 (Black
--   Friday peak) | 2017-12 5,673
-- Note the QUOTES around '2017': strftime() returns TEXT, not a number.
SELECT strftime('%Y-%m', order_purchase_timestamp) AS month,
       COUNT(*) AS orders
FROM orders
WHERE strftime('%Y', order_purchase_timestamp) = '2017'
GROUP BY month
ORDER BY month

---
## 🤖 Working with DeepSeek: the prompt-then-verify protocol

Wednesday introduced DeepSeek as a drafting tool and attached one non-negotiable condition
to it: **never trust a generated query until you have checked its number against a value
this curriculum has verified.** Date functions are where that discipline earns its keep,
because date bugs are the quietest bugs in SQL. A query that filters on the wrong quoting,
or subtracts two timestamps without `julianday()`, does not raise an error. It returns a
tidy table with a plausible-looking number in it, and you put that number in a report.

**The protocol, unchanged from Wednesday:**

1. **State the question in plain English first.** If you cannot write the question down, you
   are not ready to prompt for the answer.
2. **Give DeepSeek what it cannot guess** — the exact table name, the exact column name, and
   the dialect. Say **SQLite** explicitly. MySQL has `YEAR()`, PostgreSQL has
   `EXTRACT(YEAR FROM ...)`, and SQLite has neither; an assistant that guesses the wrong
   dialect will hand you a query that fails on `no such function: YEAR`.
3. **Run it and check the number.** Against a value you already know is right.
4. **Be able to explain every line.** If a format code is unfamiliar, ask what it does
   *before* you use the output. You will be asked to explain your queries in class.

**A good prompt for today's material:**

```
I am using SQLite. I have a table `orders` with a TEXT column
`order_purchase_timestamp` in the format 'YYYY-MM-DD HH:MM:SS'.
Write a query that counts how many orders were placed in the year 2017.
Return a single row with one column.
```

Run that prompt now, paste whatever comes back into a scratch cell, and run it. Then
compare it with the cell below — which is step 3 of the protocol, not decoration. Section 2
already established that 2017 holds **45,101** orders. Any other answer means the generated
query is wrong: most likely it compared against the number `2017` instead of the string
`'2017'` (which silently matches **zero** rows), or it reached for a `YEAR()` function that
does not exist in SQLite.

In [ ]:
%%sql
-- Step 3 of the protocol: verify before you trust. This is the same 2017 total
-- that section 2 produced, asked as a single scalar so it is easy to compare
-- against whatever DeepSeek hands you.
SELECT COUNT(*) AS orders_2017   -- Expected: 45,101
FROM orders
WHERE strftime('%Y', order_purchase_timestamp) = '2017'

## 4. Measuring a duration — `julianday()` and delivery time

Extracting a part of a date answers *when*. The other half of date work is *how long*, and
that means subtracting one timestamp from another. You cannot do that directly, because the
timestamps are text and subtracting text in SQLite produces something worse than an error
(section 6 shows exactly what).

`julianday()` is the bridge. It converts a timestamp into a single number — the count of
days elapsed since a fixed reference point far back in history — so
`julianday('2017-11-26') - julianday('2017-11-24')` is plain arithmetic that evaluates to
`2.0`. Because the conversion keeps the time of day, the result is fractional: an order
placed at 09:00 and delivered at 21:00 twelve days later comes out as `12.5`, not `12`.
Wrap the average in `ROUND(..., 1)` and you have a number fit for a slide.

Two filters in the query below are doing real work. `order_status = 'delivered'` keeps
canceled and in-flight orders out of a delivery-time average, and
`order_delivered_customer_date IS NOT NULL` guards against the handful of orders that are
marked delivered but carry no delivery timestamp. `AVG()` would have skipped those rows
silently anyway — it ignores `NULL` rather than complaining — and that is precisely why the
filter is written explicitly: it makes the decision visible to whoever reads the query next,
instead of leaving it to a function's default behaviour.

**12.6 days** is the answer, and it is a genuinely important number for Olist. It is not a
courier's transit time; it is the customer's whole wait, from the moment they click *buy* to
the moment the parcel is in their hands.

In [ ]:
%%sql
-- Average delivery time in days, for delivered orders only.
-- julianday() turns each timestamp into a number of days, so the two can be
-- subtracted. The result is fractional, hence ROUND(..., 1).
SELECT ROUND(AVG(
           julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp)
       ), 1) AS avg_delivery_days   -- Expected: 12.6
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_status = 'delivered'

## 5. Promise versus reality — comparing two date columns

The `orders` table holds a column most e-commerce datasets do not:
`order_estimated_delivery_date`, the date the customer was shown at checkout. Sitting beside
`order_delivered_customer_date`, it lets you measure something no average can capture —
not *how long delivery took*, but **how often the business broke its own promise**.

The comparison itself needs no function at all. Because both columns are ISO-formatted text,
`order_delivered_customer_date > order_estimated_delivery_date` compares them
alphabetically, and alphabetical order is chronological order — so the expression means
exactly "delivered after the promised date". This is the one place where SQLite storing
dates as text works entirely in your favour.

The answer is **7,826 late orders**, and the percentage beside it is the number that
actually travels: **8.1% of delivered orders**. Note how that percentage is built. The
denominator is not `COUNT(*)` — inside a query already filtered to late orders, `COUNT(*)`
*is* the numerator. It has to come from a separate `(SELECT COUNT(*) ...)` over all
delivered orders. And `* 1.0` is there for the reason Wednesday established: without it,
`7826 * 100 / 96478` is integer division and reports a flat `8`, not `8.1`.

Hold both numbers together, because they tell one story. Average delivery is 12.6 days and
roughly one delivery in twelve arrives later than promised — so the estimate Olist shows at
checkout is broadly honest, but not reliably so. That gap is a customer-experience problem
with a measurable size, which is the most useful kind.

In [ ]:
%%sql
-- Late deliveries: actual delivery later than the date promised at checkout.
-- Both columns are ISO text, so a plain > comparison is already chronological.
-- The denominator must come from a separate subquery over ALL delivered orders,
-- and * 1.0 forces REAL division so the percentage is not truncated to 8.
SELECT COUNT(*) AS late_orders,          -- Expected: 7,826
       ROUND(COUNT(*) * 1.0 * 100
             / (SELECT COUNT(*) FROM orders WHERE order_status = 'delivered'),
             1) AS pct_of_delivered      -- Expected: 8.1
FROM orders
WHERE order_delivered_customer_date > order_estimated_delivery_date
  AND order_status = 'delivered'

## Going deeper — when `CASE WHEN` meets `julianday()`

An average is a single number standing in for a distribution, and it hides everything
interesting about that distribution. "12.6 days" is compatible with almost every order
arriving in twelve or thirteen days, and it is equally compatible with half arriving in
four days and half in twenty-one. Those two businesses have entirely different problems, and
the average cannot tell them apart.

This is where the two days of Week 4 fuse into one technique. `julianday()` produces a
continuous number of days; Wednesday's `CASE WHEN` bins a continuous number into named
bands. Put the date arithmetic *inside* the `CASE` and you get a delivery-speed distribution
instead of a delivery-speed average — the same move as Wednesday's payment-value bands, with
a computed expression in place of a stored column.

Wednesday's band rules apply here unchanged, because they are rules about `CASE`, not about
payments: branches are checked **top to bottom and the first true one wins**, so writing the
narrow band first means each branch only needs its upper bound; and an explicit `ELSE`
catches everything the bands missed rather than leaking those rows into a `NULL` bucket.

One honest note on reading the output: these band counts are not stated anywhere in the
curriculum, so treat them as something the query is showing you rather than a target to hit.
What *is* worth noticing is the shape — the bands are far more spread out than a single
average would ever suggest, and the slowest band is not empty.

In [ ]:
%%sql
-- Wednesday's CASE WHEN + today's julianday(): a delivery-speed distribution.
-- The CASE bins a COMPUTED value (days elapsed) rather than a stored column.
-- Bands are checked in order, so each WHEN only needs its upper bound.
-- No curriculum-verified counts here — read the SHAPE, not a target number.
SELECT CASE
           WHEN julianday(order_delivered_customer_date)
                - julianday(order_purchase_timestamp) <= 7  THEN 'Fast (<= 7 days)'
           WHEN julianday(order_delivered_customer_date)
                - julianday(order_purchase_timestamp) <= 14 THEN 'Standard (8-14 days)'
           WHEN julianday(order_delivered_customer_date)
                - julianday(order_purchase_timestamp) <= 30 THEN 'Slow (15-30 days)'
           ELSE 'Very Slow (> 30 days)'
       END AS delivery_speed,
       COUNT(*) AS order_count
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
GROUP BY delivery_speed
ORDER BY order_count DESC

## Common mistakes

Every mistake below returns a result. None of them raises an error. That is what makes date
handling in SQLite worth this much attention.

**Mistake 1 — comparing a `strftime()` result to a number.** `strftime()` returns TEXT, so
`WHERE strftime('%Y', order_purchase_timestamp) = 2017` compares the string `'2017'` against
the integer `2017`. In SQLite those are different types and never equal, so the condition is
false for every row. The query succeeds and reports **0 orders in 2017** — a year you have
just seen holds 45,101 of them. The fix is one pair of quotes: `= '2017'`.

**Mistake 2 — subtracting timestamps without `julianday()`.** Writing
`order_delivered_customer_date - order_purchase_timestamp` looks like it should work, and
SQLite obligingly does *something*: it coerces each string to a number by reading digits
from the front until it hits a non-digit, so `'2017-11-24 09:22:16'` becomes `2017`. Both
sides collapse to a year, the subtraction returns a near-zero figure, and an average
delivery time of **12.6 days** silently becomes about **0.03**. Always convert first with
`julianday()`.

**Mistake 3 — testing a missing date with `= NULL`.** Unchanged from Wednesday, and it bites
just as hard on date columns: `WHERE order_delivered_customer_date = NULL` matches nothing,
because a comparison against `NULL` is *unknown*, never true. Use `IS NULL` / `IS NOT NULL`.
The count of undelivered orders is one of the first sanity checks anyone runs on this table,
and this typo turns it into a confident zero.

The correct query below fixes all three at once. Notice where the missing-date test now
sits: inside a `CASE` rather than in the `WHERE`, so the guard applies to the average alone
and the order count still covers the whole of 2017 — **45,101**, the number section 2
verified. Pushing a guard into the `WHERE` clause silently narrows every other column in the
query too, which is its own quiet mistake.

In [ ]:
%%sql
-- ── COMMON MISTAKES ────────────────────────────────────────────────
-- WRONG 1 — strftime() returns TEXT; comparing it to the NUMBER 2017 is never
-- true, so this reports 0 orders for a year that holds 45,101:
--   WHERE strftime('%Y', order_purchase_timestamp) = 2017
--
-- WRONG 2 — subtracting text timestamps does not error; SQLite reads digits
-- off the front of each string, both become 2017, and 12.6 days silently
-- becomes about 0.03:
--   AVG(order_delivered_customer_date - order_purchase_timestamp)
--
-- WRONG 3 — '= NULL' is never true, so this matches zero rows:
--   WHERE order_delivered_customer_date = NULL
--
-- CORRECT — all three fixes in one query: quoted '2017', julianday() before
-- subtracting, and IS NULL / IS NOT NULL for the missing-date tests. The
-- missing-date guard lives inside a CASE (with an explicit ELSE NULL) rather
-- than in the WHERE, so the order count still covers ALL of 2017.
SELECT COUNT(*) AS orders_2017,          -- Expected: 45,101
       SUM(CASE WHEN order_delivered_customer_date IS NULL
                THEN 1 ELSE 0 END) AS no_delivery_date,
       ROUND(AVG(CASE WHEN order_delivered_customer_date IS NOT NULL
                      THEN julianday(order_delivered_customer_date)
                           - julianday(order_purchase_timestamp)
                      ELSE NULL END), 1) AS avg_delivery_days_2017
FROM orders
WHERE strftime('%Y', order_purchase_timestamp) = '2017'

## Mini-challenge — your turn

⏱ ~5–10 minutes

Section 3 showed the monthly shape of 2017 and found the Black Friday spike. Operations now
wants the same view for **2018**, with one extra column: for each month, how many orders,
**and** the average delivery time in days for the orders placed that month.

Build it from pieces you have already run today:

1. `strftime('%Y-%m', order_purchase_timestamp) AS month` for the grouping label.
2. A `WHERE` clause filtering to 2018 — remember the **quotes** around `'2018'`.
3. `COUNT(*)` for the order count.
4. `ROUND(AVG(julianday(...) - julianday(...)), 1)` for the average delivery days.
5. `GROUP BY month ORDER BY month`.

**Expected:** one row per month of 2018, in chronological order, and the order counts must
sum to **54,011** — the 2018 total from section 2. If your months do not add up to that, the
`WHERE` filter is wrong. 2018 is also a partial year at its far end, so expect the last
months to look thin; that is the section 2 lesson repeating itself.

*Stretch, if you finish early:* add a third column counting the late orders in each month
with `SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1
ELSE 0 END)`, and see whether the lateness rate rises in the busiest months.

In [ ]:
%%sql
-- ⏱ ~5-10 min — your turn! Replace the placeholder below with your own query.
SELECT 'write your query here' AS todo

## Session Summary

| Function / idea | What it does | Example |
|---|---|---|
| dates are `TEXT` | SQLite has no date type; timestamps are ISO strings, so text order = time order | `typeof(order_purchase_timestamp)` → `text` |
| `strftime('%Y', col)` | pulls the 4-digit year out of a timestamp, as TEXT | `strftime('%Y', order_purchase_timestamp) AS year` |
| `strftime('%Y-%m', col)` | year and month together — keeps Novembers of different years apart | `GROUP BY strftime('%Y-%m', ...)` |
| other format codes | `%d` day, `%H` hour, `%w` weekday (0 = Sunday … 6 = Saturday) | `strftime('%w', order_purchase_timestamp)` |
| quoting the result | `strftime()` returns TEXT — compare to `'2017'`, never `2017` | `WHERE strftime('%Y', ...) = '2017'` |
| `GROUP BY <extracted part>` | turns individual timestamps into a trend | `GROUP BY year` → 2016 329, 2017 45,101, 2018 54,011 |
| `julianday(col)` | converts a timestamp to a number of days so dates can be subtracted | `julianday(a) - julianday(b)` |
| fractional days | the result keeps the time of day, so round it for reporting | `ROUND(AVG(...), 1)` → 12.6 |
| comparing two date columns | `>` on two ISO text columns is already chronological | `delivered > estimated` → 7,826 late orders |
| `* 1.0` | forces REAL division so a rate is not truncated to a whole number | `7,826 → 8.1%` of delivered |
| `CASE WHEN` + `julianday()` | bins a computed duration into named speed bands | `WHEN ... <= 7 THEN 'Fast'` |

**The three questions to ask of every date query you write:** is the value I am comparing
against quoted as text; did I convert with `julianday()` before subtracting; and is the time
window I am reporting on actually complete?

---
### Closing the session — the group exercise

Today's session ends with a **group exercise** that puts Wednesday's `CASE WHEN` and
today's date functions to work on the same four questions. In your groups you will:

1. **Classify customers into geographic regions** with `CASE WHEN` on `customer_state` —
   Southeast (SP, RJ, MG, ES), South (RS, SC, PR), Northeast (BA, CE, PE, MA, RN, PB, AL,
   SE, PI), and Other for everything else.
2. **Compare weekend against weekday order volume**, using `strftime('%w', ...)` — which
   returns `0` for Sunday through `6` for Saturday, so a `CASE WHEN` over those two values
   splits the week in two.
3. **Rank average delivery time by customer state**, showing the five fastest and the five
   slowest — `julianday()` for the duration, a join to `customers` for the state.
4. **Build a month-by-month view of 2018** with order count, total revenue from
   `order_payments`, and a `CASE WHEN` flag marking each month a "peak month" at 6,000
   orders or more.

None of these four has a published answer for you to check against, which is deliberate —
the verification is your own. Sanity-check each result against a number you already trust:
the regions in question 1 must sum to 99,441 customers, and the months in question 4 must
sum to 54,011 orders. If a total does not reconcile, the query is wrong before the
interpretation ever begins.

---
**Coming up next week — Week 5: subqueries and window functions.** You met your first
subquery today, tucked into the late-delivery percentage:
`(SELECT COUNT(*) FROM orders WHERE order_status = 'delivered')`, a query used as a single
value inside another query. Next week that becomes the main subject — using one query's
result inside another to answer questions a single `GROUP BY` cannot reach, such as which
orders sit above the average payment value, and then window functions, which let you rank
and total *across* rows without collapsing them.